# Learned Stage-Wise Curvature Tuning on DTD

This notebook evaluates Learned Stage-Wise Curvature Tuning (SW-CT) on the Describable Textures Dataset (DTD) using an ImageNet-pretrained ResNet-50.

DTD is a texture-recognition dataset, making it different from the Beans dataset used in the previous experiment. Testing on DTD helps determine whether the behavior of Learned SW-CT generalizes to a different visual classification problem.

The experiments compare four conditions:

1. **Baseline** — the original pretrained ResNet with ReLU activations and a trained downstream classifier.
2. **Single-Parameter Curvature Tuning (S-CT)** — one global beta selected through validation-based exhaustive search.
3. **Learned SW-CT with fixed initialization** — four trainable stage-wise beta parameters initialized at 0.78.
4. **Learned SW-CT with random initialization** — four trainable stage-wise beta parameters independently initialized between 0.70 and 0.99.

Each condition is evaluated using seeds 42, 43, and 44.

Experiments are conducted for both 20 and 30 epochs to examine whether additional optimization improves accuracy or changes the learned stage-wise beta values.

Wall-clock execution time is recorded to compare the computational cost of Baseline, S-CT, and Learned SW-CT.

In [1]:
!nvidia-smi

Fri Oct  2 09:11:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%cd /content

!git clone -b stage-wise-ct \
    https://github.com/Syed1611/curvature-tuning-research.git

%cd /content/curvature-tuning-research/src/curvature-tuning

!git branch --show-current
!git log -1 --oneline

## Environment Setup

The repository dependencies are installed before running the experiments.

After installation, the Colab runtime should be restarted so that the required package versions are loaded correctly.

In [ ]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

In [1]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
import torch

os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/content/curvature-tuning-research/src/curvature-tuning
Working directory: /content/curvature-tuning-research/src/curvature-tuning
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## DTD Dataset

Before running the experiments, the DTD data loaders are checked to confirm that the downstream dataset can be loaded correctly.

The same dataset split and random seed protocol are used for Baseline, S-CT, and Learned SW-CT so that the methods are compared under the same conditions.

In [2]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_dtd",
    train_batch_size=32,
    test_batch_size=128,
    seed=42,
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

100%|██████████| 625M/625M [00:17<00:00, 36.4MB/s] 


Train samples: 1880
Validation samples: 1880
Test samples: 1880
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels: tensor([12, 28, 12, 14, 24, 10,  5, 39, 17, 17])


# ResNet-50 Experiments

ResNet-50 is evaluated first because it was used during the initial development of Learned SW-CT.

The experiment compares Baseline, S-CT, fixed-initialization SW-CT, and random-initialization SW-CT.

All methods use the same ImageNet-pretrained backbone, DTD dataset split, experimental seeds, and downstream training protocol.

## ResNet-50 — 20-Epoch Evaluation

The first evaluation uses 20 training epochs.

This provides the primary reference experiment and maintains compatibility with the earlier Learned SW-CT work.

### Methods

**Baseline**

The pretrained ResNet-50 retains its original ReLU activations. The backbone is frozen and only the downstream classifier is trained.

**S-CT**

The original Single-Parameter Curvature Tuning method searches over a single beta shared throughout the network. Candidate beta values from 0.70 to 1.00 are evaluated using validation performance, and the best candidate is evaluated on the test set.

The recorded S-CT execution time includes the complete beta search rather than only the final selected beta.

**Learned SW-CT — Fixed Initialization**

Four trainable beta parameters are assigned to the four ResNet stages. All four begin at beta = 0.78.

This preserves the initialization used in the original Learned SW-CT experiments.

**Learned SW-CT — Random Initialization**

Four trainable stage-wise beta parameters are initialized independently between 0.70 and 0.99.

The random initialization is generated deterministically from the experiment seed. This allows the experiment to test whether Learned SW-CT depends on the fixed 0.78 starting value while remaining reproducible.

In [3]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-02 09:13:38.103 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_dtd_resnet50_seed42_epochs20.log
2026-10-02 09:13:38.105 | INFO     | __main__:main:78 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 189MB/s]
2026-10-02 09:13:40.012 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 09:13:40.056 | INFO     | __main__:main:116 - Number of trainable parameters: 96303
2026-10-02 09:13:40.056 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 09:13:57.735 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.923605, Accuracy: 0.00%
2026-10-02 09:13:58.035 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 2.232503, Val Accuracy: 51.81%
2026-10-02 09:13:58.036 | INFO     | train:linear_probe:190 - New best validation accuracy: 51.81 at 

In [4]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-02 09:32:14.621 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_dtd_resnet50_seed43_epochs20.log
2026-10-02 09:32:14.623 | INFO     | __main__:main:78 - Running on cuda
2026-10-02 09:32:15.384 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 09:32:15.421 | INFO     | __main__:main:116 - Number of trainable parameters: 96303
2026-10-02 09:32:15.421 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 09:32:30.793 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.878620, Accuracy: 3.12%
2026-10-02 09:32:31.100 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 2.208647, Val Accuracy: 55.00%
2026-10-02 09:32:31.101 | INFO     | train:linear_probe:190 - New best validation accuracy: 55.00 at epoch 1
2026-10-02 09:32:31.147 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 2.056217, Accuracy: 59.38%
2026-10-02 09:32:31.445 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 1.453257, Val Accu

In [5]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-02 09:50:52.979 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_dtd_resnet50_seed44_epochs20.log
2026-10-02 09:50:52.981 | INFO     | __main__:main:78 - Running on cuda
2026-10-02 09:50:53.742 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 09:50:53.777 | INFO     | __main__:main:116 - Number of trainable parameters: 96303
2026-10-02 09:50:53.777 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 09:51:09.490 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.986091, Accuracy: 0.00%
2026-10-02 09:51:09.815 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 2.241592, Val Accuracy: 52.13%
2026-10-02 09:51:09.816 | INFO     | train:linear_probe:190 - New best validation accuracy: 52.13 at epoch 1
2026-10-02 09:51:09.861 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 2.147360, Accuracy: 46.88%
2026-10-02 09:51:10.180 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 1.492244, Val Accu

### Learned SW-CT with Fixed Beta Initialization

The original Learned SW-CT configuration is first reproduced using the fixed initialization:

\[[\beta_1,\beta_2,\beta_3,\beta_4]=[0.78,0.78,0.78,0.78].\]

The initial value is only a starting point. During training, the four beta parameters are optimized independently together with the downstream classifier.

These runs provide the reference against which random initialization will be compared.

In [6]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-02 10:10:01.559 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet50_seed42_epochs20_initfixed0.78_betalr0.01.log
2026-10-02 10:10:01.617 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 10:10:02.373 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-02 10:10:02.373 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-02 10:10:02.426 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-02 10:10:02.427 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 10:10:02.427 | INFO     | __main__:main:439 - Total trainable parameters: 96307
2026-10-02 10:10:04.164 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.877501, Accuracy: 3.12%
2026-10-02 10:10:54.355 | INFO     | train:test_epoch:90 - E

In [7]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-02 10:26:57.225 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet50_seed43_epochs20_initfixed0.78_betalr0.01.log
2026-10-02 10:26:57.280 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 10:26:58.107 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-02 10:26:58.108 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-02 10:26:58.181 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-02 10:26:58.182 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 10:26:58.183 | INFO     | __main__:main:439 - Total trainable parameters: 96307
2026-10-02 10:26:59.955 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.018456, Accuracy: 0.00%
2026-10-02 10:27:49.366 | INFO     | train:test_epoch:90 - E

In [8]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-02 10:44:04.982 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet50_seed44_epochs20_initfixed0.78_betalr0.01.log
2026-10-02 10:44:05.055 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 10:44:06.087 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-02 10:44:06.087 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-02 10:44:06.171 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-02 10:44:06.172 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 10:44:06.172 | INFO     | __main__:main:439 - Total trainable parameters: 96307
2026-10-02 10:44:08.062 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.083963, Accuracy: 0.00%
2026-10-02 10:44:57.824 | INFO     | train:test_epoch:90 - E

### Learned SW-CT with Random Beta Initialization

The fixed beta = 0.78 initialization may raise the question of whether Learned SW-CT performs well only because its starting point was informed by the earlier S-CT experiments.

To test this dependency, each of the four stage-wise beta parameters is now initialized independently from a uniform distribution between 0.70 and 0.99.

The initialization is reproducible: the random generator is seeded using the experiment seed. Therefore, each seed always receives the same initial beta vector.

If random initialization produces similar performance to the fixed 0.78 experiment, this provides evidence that the Learned SW-CT result is not dependent on a single hand-selected initialization.

In [9]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-02 11:01:09.349 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet50_seed42_epochs20_initrandom_betalr0.01.log
2026-10-02 11:01:09.441 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 11:01:10.510 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-02 11:01:10.510 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-02 11:01:10.594 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-02 11:01:10.595 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 11:01:10.595 | INFO     | __main__:main:439 - Total trainable parameters: 96307
2026-10-02 11:01:12.439 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.900538, Accuracy: 6.25%
2026-10-02 11:02:02.362 | INFO     | train:test_

In [10]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-02 11:18:15.242 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet50_seed43_epochs20_initrandom_betalr0.01.log
2026-10-02 11:18:15.343 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 11:18:16.419 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-02 11:18:16.420 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-02 11:18:16.501 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-02 11:18:16.502 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 11:18:16.502 | INFO     | __main__:main:439 - Total trainable parameters: 96307
2026-10-02 11:18:18.321 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.011998, Accuracy: 0.00%
2026-10-02 11:19:07.425 | INFO     | train:tes

In [11]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-02 11:35:15.206 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet50_seed44_epochs20_initrandom_betalr0.01.log
2026-10-02 11:35:15.276 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 11:35:16.164 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-02 11:35:16.165 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-02 11:35:16.227 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-02 11:35:16.228 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 11:35:16.228 | INFO     | __main__:main:439 - Total trainable parameters: 96307
2026-10-02 11:35:17.765 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.039367, Accuracy: 0.00%
2026-10-02 11:36:07.371 | INFO     | train:tes

# ResNet-50 — 30-Epoch Evaluation

The same experiments are now repeated for 30 epochs.

The purpose of increasing the training budget is to determine whether the 20-epoch results are limited by insufficient optimization.

If a method improves substantially with 30 epochs, this suggests that additional optimization is useful. If performance remains similar, it indicates that the method has largely stabilized within the original 20-epoch training budget.

All other experimental settings remain unchanged.

For random-initialization SW-CT, the same seed produces exactly the same initial four-beta vector in both the 20-epoch and 30-epoch experiments. Therefore, initialization is held constant while the training duration changes.

Execution time is again recorded to measure the computational cost of extending training from 20 to 30 epochs.

In [12]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-02 11:52:17.951 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_dtd_resnet50_seed42_epochs30.log
2026-10-02 11:52:17.953 | INFO     | __main__:main:78 - Running on cuda
2026-10-02 11:52:18.982 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 11:52:19.019 | INFO     | __main__:main:116 - Number of trainable parameters: 96303
2026-10-02 11:52:19.020 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 11:52:41.394 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.923605, Accuracy: 0.00%
2026-10-02 11:52:41.845 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 2.232503, Val Accuracy: 51.81%
2026-10-02 11:52:41.846 | INFO     | train:linear_probe:190 - New best validation accuracy: 51.81 at epoch 1
2026-10-02 11:52:41.967 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 1.989952, Accuracy: 68.75%
2026-10-02 11:52:42.451 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 1.498326, Val Accu

In [13]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-02 12:17:51.834 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_dtd_resnet50_seed43_epochs30.log
2026-10-02 12:17:51.837 | INFO     | __main__:main:78 - Running on cuda
2026-10-02 12:17:53.050 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 12:17:53.118 | INFO     | __main__:main:116 - Number of trainable parameters: 96303
2026-10-02 12:17:53.119 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 12:18:19.572 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.878620, Accuracy: 3.12%
2026-10-02 12:18:20.120 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 2.208647, Val Accuracy: 55.00%
2026-10-02 12:18:20.121 | INFO     | train:linear_probe:190 - New best validation accuracy: 55.00 at epoch 1
2026-10-02 12:18:20.237 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 2.056217, Accuracy: 59.38%
2026-10-02 12:18:20.787 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 1.453257, Val Accu

In [14]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-02 12:43:43.995 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_dtd_resnet50_seed44_epochs30.log
2026-10-02 12:43:43.997 | INFO     | __main__:main:78 - Running on cuda
2026-10-02 12:43:45.184 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 12:43:45.242 | INFO     | __main__:main:116 - Number of trainable parameters: 96303
2026-10-02 12:43:45.243 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 12:44:08.260 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.986091, Accuracy: 0.00%
2026-10-02 12:44:08.794 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 2.241592, Val Accuracy: 52.13%
2026-10-02 12:44:08.796 | INFO     | train:linear_probe:190 - New best validation accuracy: 52.13 at epoch 1
2026-10-02 12:44:08.919 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 2.147360, Accuracy: 46.88%
2026-10-02 12:44:09.441 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 1.492244, Val Accu

In [15]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-02 13:09:15.684 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet50_seed42_epochs30_initfixed0.78_betalr0.01.log
2026-10-02 13:09:15.775 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 13:09:16.679 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-02 13:09:16.679 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-02 13:09:16.764 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-02 13:09:16.765 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 13:09:16.765 | INFO     | __main__:main:439 - Total trainable parameters: 96307
2026-10-02 13:09:18.691 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.877501, Accuracy: 3.12%
2026-10-02 13:10:07.494 | INFO     | train:test_epoch:90 - E

In [16]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-02 13:34:39.511 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet50_seed43_epochs30_initfixed0.78_betalr0.01.log
2026-10-02 13:34:39.582 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 13:34:40.516 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-02 13:34:40.516 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-02 13:34:40.587 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-02 13:34:40.587 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 13:34:40.588 | INFO     | __main__:main:439 - Total trainable parameters: 96307
2026-10-02 13:34:42.136 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.018456, Accuracy: 0.00%
2026-10-02 13:35:31.886 | INFO     | train:test_epoch:90 - E

In [17]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-02 14:00:11.092 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet50_seed44_epochs30_initfixed0.78_betalr0.01.log
2026-10-02 14:00:11.193 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 14:00:12.325 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-02 14:00:12.325 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-02 14:00:12.406 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-02 14:00:12.408 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 14:00:12.408 | INFO     | __main__:main:439 - Total trainable parameters: 96307
2026-10-02 14:00:14.273 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.083963, Accuracy: 0.00%
2026-10-02 14:01:04.009 | INFO     | train:test_epoch:90 - E

In [18]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-02 14:25:45.751 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet50_seed42_epochs30_initrandom_betalr0.01.log
2026-10-02 14:25:45.844 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 14:25:46.942 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-02 14:25:46.943 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-02 14:25:47.025 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-02 14:25:47.026 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 14:25:47.027 | INFO     | __main__:main:439 - Total trainable parameters: 96307
2026-10-02 14:25:48.913 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.900538, Accuracy: 6.25%
2026-10-02 14:26:38.805 | INFO     | train:test_

In [19]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-02 14:51:14.927 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet50_seed43_epochs30_initrandom_betalr0.01.log
2026-10-02 14:51:14.985 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 14:51:15.895 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-02 14:51:15.895 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-02 14:51:15.967 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-02 14:51:15.968 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 14:51:15.968 | INFO     | __main__:main:439 - Total trainable parameters: 96307
2026-10-02 14:51:17.772 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.011998, Accuracy: 0.00%
2026-10-02 14:52:07.543 | INFO     | train:tes

In [20]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-02 15:16:40.123 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet50_seed44_epochs30_initrandom_betalr0.01.log
2026-10-02 15:16:40.189 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 15:16:41.139 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-02 15:16:41.139 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-02 15:16:41.219 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-02 15:16:41.220 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 15:16:41.220 | INFO     | __main__:main:439 - Total trainable parameters: 96307
2026-10-02 15:16:43.062 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.039367, Accuracy: 0.00%
2026-10-02 15:17:32.870 | INFO     | train:tes